# Xử lý dữ liệu thiếu và gộp dữ liệu tai nạn giao thông 2024

Notebook này đọc ba file gốc `collision`, `vehicle`, `casualty`, xử lý dữ liệu thiếu theo mục tiêu **trực quan hóa dữ liệu/BI**, rồi gộp thành một file duy nhất.

## Nguyên tắc xử lý

Không coi mọi giá trị thiếu là cùng một loại. Notebook phân biệt:

1. **Unknown / No data** – thông tin đáng lẽ có thể tồn tại nhưng nguồn không biết/không ghi nhận.
2. **Not applicable** – thuộc tính không áp dụng cho bản ghi đó theo logic nghiệp vụ.
3. **Structural / specification missing** – biến cũ/lịch sử hoặc biến bị ảnh hưởng bởi chuyển đổi đặc tả; không nội suy.
4. **Merge-generated missing** – ô trống sinh ra do phép gộp, ví dụ vehicle không có casualty; không phải lỗi chất lượng dữ liệu.

Quy tắc thực thi:

- Mã `-1` được chuyển thành `NA` ở tầng làm sạch để có thể phân loại lý do thiếu.
- Biến phân loại thiếu thật được gán **`Không xác định`**.
- Chỉ gán **`Không áp dụng`** khi có quy tắc nghiệp vụ đủ rõ.
- Biến số liên tục giữ `NA`; **không điền mean/median/mode**. Thêm cột `_was_missing`, `_status` và khi cần `_is_not_applicable`.
- Các cột lịch sử/dư thừa và `special_conditions_at_site` được loại khỏi bảng phục vụ trực quan hóa, nhưng vẫn tồn tại trong dữ liệu nguồn.
- Các cột enhanced severity giữ `NA` và có cột trạng thái để người dùng nhìn thấy coverage.
- Sau khi gộp, thêm `collision_count_flag`, `vehicle_count_flag`, `casualty_count_flag` để tránh double counting trong Power BI/Tableau.


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_rows", 140)

UNKNOWN_LABEL = "Không xác định"
NOT_APPLICABLE_LABEL = "Không áp dụng"
OBSERVED_LABEL = "Có dữ liệu"

FILE_NAMES = {
    "collision": "dft-road-casualty-statistics-collision-2024.csv",
    "vehicle": "dft-road-casualty-statistics-vehicle-2024.csv",
    "casualty": "dft-road-casualty-statistics-casualty-2024.csv",
}

# Hỗ trợ chạy tại thư mục dự án, thư mục upload hoặc Google Colab.
candidates = [
    Path.cwd(),
    Path.cwd() / "upload",
    Path.cwd().parent / "upload",
    Path.cwd().parent.parent / "upload",
    Path("/content"),
]
DATA_DIR = next(
    (folder for folder in candidates if all((folder / name).exists() for name in FILE_NAMES.values())),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Không tìm thấy đủ ba file CSV. Hãy đặt chúng cùng thư mục với notebook "
        "hoặc trong thư mục upload."
    )

OUTPUT_CSV = Path.cwd() / "du_lieu_tai_nan_2024_san_sang_truc_quan.csv"
print(f"Dữ liệu nguồn: {DATA_DIR.resolve()}")
print(f"Dữ liệu đầu ra: {OUTPUT_CSV.resolve()}")


Dữ liệu nguồn: D:\
Dữ liệu đầu ra: D:\du_lieu_tai_nan_2024_san_sang_truc_quan.csv


## 1. Đọc ba bảng gốc

Các cột định danh được đọc dưới dạng chuỗi để giữ nguyên số 0 ở đầu.

In [3]:
identifier_dtypes = {
    "collision_index": "string",
    "collision_ref_no": "string",
}

collision_raw = pd.read_csv(
    DATA_DIR / FILE_NAMES["collision"], dtype=identifier_dtypes, low_memory=False
)
vehicle_raw = pd.read_csv(
    DATA_DIR / FILE_NAMES["vehicle"], dtype=identifier_dtypes, low_memory=False
)
casualty_raw = pd.read_csv(
    DATA_DIR / FILE_NAMES["casualty"], dtype=identifier_dtypes, low_memory=False
)

input_summary = pd.DataFrame({
    "Bảng": ["Collision", "Vehicle", "Casualty"],
    "Số dòng": [len(collision_raw), len(vehicle_raw), len(casualty_raw)],
    "Số cột": [collision_raw.shape[1], vehicle_raw.shape[1], casualty_raw.shape[1]],
    "Dòng trùng hoàn toàn": [
        collision_raw.duplicated().sum(),
        vehicle_raw.duplicated().sum(),
        casualty_raw.duplicated().sum(),
    ],
})
display(input_summary)

,Bảng,Số dòng,Số cột,Dòng trùng hoàn toàn
0,Collision,100927,44,0
1,Vehicle,183514,32,0
2,Casualty,128272,23,0


## 2. Chuẩn hóa dữ liệu và nhận diện mã thiếu ban đầu

Ở bước này, mã `-1`/chuỗi `"-1"` chỉ được chuyển thành `NA`. **Chưa** gán ngay thành `Không xác định`, vì cùng một `NA` có thể mang ý nghĩa khác nhau tùy biến và ngữ cảnh.

Việc phân loại `Unknown`, `Not applicable` và `Structural missing` được thực hiện ở các bước sau.


In [4]:
MISSING_CODE_COLUMNS = {
    "collision": [
        "local_authority_district", "speed_limit", "junction_detail_historic",
        "junction_detail", "junction_control", "second_road_class",
        "second_road_number", "pedestrian_crossing_human_control_historic",
        "pedestrian_crossing_physical_facilities_historic", "pedestrian_crossing",
        "light_conditions", "road_surface_conditions", "special_conditions_at_site",
        "carriageway_hazards_historic", "carriageway_hazards", "trunk_road_flag",
        "enhanced_severity_collision",
    ],
    "vehicle": [
        "vehicle_type", "towing_and_articulation", "vehicle_manoeuvre_historic",
        "vehicle_manoeuvre", "vehicle_direction_from", "vehicle_direction_to",
        "vehicle_location_restricted_lane_historic", "vehicle_location_restricted_lane",
        "junction_location", "skidding_and_overturning", "hit_object_in_carriageway",
        "vehicle_leaving_carriageway", "hit_object_off_carriageway",
        "first_point_of_impact", "vehicle_left_hand_drive",
        "journey_purpose_of_driver_historic", "journey_purpose_of_driver",
        "sex_of_driver", "age_of_driver", "age_band_of_driver",
        "engine_capacity_cc", "propulsion_code", "age_of_vehicle",
        "driver_imd_decile", "driver_distance_banding",
    ],
    "casualty": [
        "sex_of_casualty", "age_of_casualty", "age_band_of_casualty",
        "pedestrian_location", "pedestrian_movement", "car_passenger",
        "bus_or_coach_passenger", "pedestrian_road_maintenance_worker",
        "casualty_type", "casualty_imd_decile", "enhanced_casualty_severity",
        "casualty_distance_banding",
    ],
}

TEXT_MISSING_COLUMNS = {
    "collision": ["lsoa_of_accident_location"],
    "vehicle": ["generic_make_model", "lsoa_of_driver"],
    "casualty": ["lsoa_of_casualty"],
}


def clean_table(dataframe, missing_code_columns, text_missing_columns):
    result = dataframe.copy()
    result.columns = (
        result.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )

    for column in result.select_dtypes(include=["object", "string"]).columns:
        result[column] = result[column].astype("string").str.strip()
        result[column] = result[column].replace("", pd.NA)

    for column in missing_code_columns:
        if column in result.columns:
            result[column] = result[column].mask(result[column].eq(-1))

    for column in text_missing_columns:
        if column in result.columns:
            result[column] = result[column].replace("-1", pd.NA)

    return result.drop_duplicates().reset_index(drop=True).convert_dtypes()


collision = clean_table(
    collision_raw, MISSING_CODE_COLUMNS["collision"], TEXT_MISSING_COLUMNS["collision"]
)
vehicle = clean_table(
    vehicle_raw, MISSING_CODE_COLUMNS["vehicle"], TEXT_MISSING_COLUMNS["vehicle"]
)
casualty = clean_table(
    casualty_raw, MISSING_CODE_COLUMNS["casualty"], TEXT_MISSING_COLUMNS["casualty"]
)

print(f"Sau làm sạch: {len(collision):,} collision, {len(vehicle):,} vehicle, {len(casualty):,} casualty")

# Cảnh báo dữ liệu nguồn: e-scooter có thể thuộc bản phát hành trước khi vehicle_type
# được tái phân loại đầy đủ. Notebook không tự sửa mã nguồn; chỉ cảnh báo để tránh
# vô tình "suy đoán" vehicle_type.
if "escooter_flag" in vehicle.columns and "vehicle_type" in vehicle.columns:
    escooter_rows = vehicle["escooter_flag"].eq(1)
    if escooter_rows.any():
        escooter_type_missing = int(vehicle.loc[escooter_rows, "vehicle_type"].isna().sum())
        print(
            f"E-scooter: {int(escooter_rows.sum()):,} dòng; "
            f"vehicle_type thiếu: {escooter_type_missing:,}. "
            "Không tự nội suy vehicle_type từ notebook."
        )


Sau làm sạch: 100,927 collision, 183,514 vehicle, 128,272 casualty
E-scooter: 1,322 dòng; vehicle_type thiếu: 1,322. Không tự nội suy vehicle_type từ notebook.


## 3. Chuẩn hóa ngày giờ và tạo biến thời gian

Các trường tháng, quý, ngày, giờ và cuối tuần được tạo để dùng trực tiếp trong biểu đồ.

In [5]:
collision["date"] = pd.to_datetime(collision["date"], dayfirst=True, errors="coerce")
collision["collision_month"] = collision["date"].dt.month.astype("Int64")
collision["collision_quarter"] = collision["date"].dt.quarter.map(
    lambda value: f"Q{value}" if pd.notna(value) else pd.NA
).astype("string")
collision["collision_day_of_month"] = collision["date"].dt.day.astype("Int64")
collision["collision_hour"] = pd.to_numeric(
    collision["time"].astype("string").str.slice(0, 2), errors="coerce"
).astype("Int64")
collision["is_weekend"] = collision["date"].dt.dayofweek.ge(5).astype("boolean")

print(f"Ngày không hợp lệ: {collision['date'].isna().sum():,}")
print(f"Khoảng ngày: {collision['date'].min().date()} đến {collision['date'].max().date()}")

Ngày không hợp lệ: 0
Khoảng ngày: 2024-01-01 đến 2024-12-31


## 4. Phân loại cách xử lý dữ liệu thiếu

| Loại thiếu | Ví dụ | Xử lý |
|---|---|---|
| **Unknown / No data** | tuổi tài xế không ghi nhận, giới tính không xác định | Categorical → `Không xác định`; numeric → giữ `NA` + status |
| **Not applicable** | `junction_control` khi collision không ở junction; `engine_capacity_cc` của pedal cycle/e-scooter | Giữ numeric `NA` hoặc gán nhãn `Không áp dụng` cho categorical + status |
| **Structural/specification missing** | cột `_historic`, `special_conditions_at_site` trong bảng harmonized | Không nội suy; loại khỏi dataset trực quan hóa chính |
| **Merge-generated missing** | casualty columns trên vehicle không có casualty | Giữ trống; dùng `record_type`, `has_casualty_record` và `casualty_data_status` |

> Không dùng mean/median/mode để “lấp” dữ liệu chỉ nhằm mục tiêu làm dashboard không còn ô trống.


In [6]:
# Các cột không đưa vào bảng phân tích harmonized.
COLLISION_DROP_COLUMNS = [
    "local_authority_district",
    "junction_detail_historic",
    "pedestrian_crossing_human_control_historic",
    "pedestrian_crossing_physical_facilities_historic",
    "carriageway_hazards_historic",
    # Biến này chịu ảnh hưởng mạnh của chuyển đổi specification;
    # ưu tiên carriageway_hazards cho bảng trực quan hóa chính.
    "special_conditions_at_site",
]
VEHICLE_DROP_COLUMNS = [
    "vehicle_manoeuvre_historic",
    "vehicle_location_restricted_lane_historic",
    "journey_purpose_of_driver_historic",
]

# Categorical mà NA thực sự được coi là Unknown sau khi đã xử lý các trường hợp N/A.
COLLISION_UNKNOWN_CATEGORICAL = [
    "junction_detail", "second_road_class", "second_road_number",
    "pedestrian_crossing", "light_conditions", "road_surface_conditions",
    "carriageway_hazards", "trunk_road_flag", "lsoa_of_accident_location",
]
VEHICLE_UNKNOWN_CATEGORICAL = [
    "vehicle_type", "towing_and_articulation", "vehicle_manoeuvre",
    "vehicle_direction_from", "vehicle_direction_to",
    "vehicle_location_restricted_lane", "junction_location",
    "skidding_and_overturning", "hit_object_in_carriageway",
    "vehicle_leaving_carriageway", "hit_object_off_carriageway",
    "first_point_of_impact", "vehicle_left_hand_drive",
    "journey_purpose_of_driver", "sex_of_driver", "age_band_of_driver",
    "generic_make_model", "driver_imd_decile", "lsoa_of_driver",
    "driver_distance_banding",
]
CASUALTY_UNKNOWN_CATEGORICAL = [
    "sex_of_casualty", "age_band_of_casualty", "pedestrian_location",
    "pedestrian_movement", "car_passenger", "bus_or_coach_passenger",
    "pedestrian_road_maintenance_worker", "casualty_type",
    "casualty_imd_decile", "lsoa_of_casualty", "casualty_distance_banding",
]


def fill_unknown_categorical(dataframe, columns):
    """Chỉ gán Unknown cho NA không có quy tắc Not applicable riêng."""
    result = dataframe.copy()
    for column in columns:
        if column not in result.columns:
            continue
        series = result[column]
        if pd.api.types.is_numeric_dtype(series):
            non_missing = pd.to_numeric(series.dropna(), errors="coerce")
            integer_like = bool(non_missing.empty or np.isclose(non_missing % 1, 0).all())
            if integer_like:
                series = pd.to_numeric(series, errors="coerce").astype("Int64")
        result[column] = series.astype("string").fillna(UNKNOWN_LABEL)
    return result


def apply_categorical_missing_reason(
    dataframe,
    column,
    not_applicable_mask=None,
):
    """
    Biến categorical: giữ giá trị quan sát; NA -> 'Không áp dụng' nếu rule đúng,
    còn lại -> 'Không xác định'. Đồng thời thêm <column>_status.
    """
    result = dataframe.copy()
    raw = result[column]
    missing = raw.isna()

    if not_applicable_mask is None:
        not_applicable_mask = pd.Series(False, index=result.index)
    else:
        not_applicable_mask = not_applicable_mask.reindex(result.index).fillna(False)

    is_na = missing & not_applicable_mask
    is_unknown = missing & ~not_applicable_mask

    display_series = raw.astype("string")
    display_series = display_series.mask(is_na, NOT_APPLICABLE_LABEL)
    display_series = display_series.mask(is_unknown, UNKNOWN_LABEL)

    status = pd.Series(OBSERVED_LABEL, index=result.index, dtype="string")
    status = status.mask(is_na, NOT_APPLICABLE_LABEL)
    status = status.mask(is_unknown, UNKNOWN_LABEL)

    result[column] = display_series
    result[f"{column}_status"] = status
    result[f"{column}_was_missing"] = missing.astype("int8")
    return result


def add_numeric_missing_reason(
    dataframe,
    column,
    not_applicable_mask=None,
):
    """
    Biến số: không nội suy. Giữ NA và tạo status để tách Unknown / Not applicable.
    """
    result = dataframe.copy()
    missing = result[column].isna()

    if not_applicable_mask is None:
        not_applicable_mask = pd.Series(False, index=result.index)
    else:
        not_applicable_mask = not_applicable_mask.reindex(result.index).fillna(False)

    is_na = missing & not_applicable_mask
    is_unknown = missing & ~not_applicable_mask

    status = pd.Series(OBSERVED_LABEL, index=result.index, dtype="string")
    status = status.mask(is_na, NOT_APPLICABLE_LABEL)
    status = status.mask(is_unknown, UNKNOWN_LABEL)

    result[f"{column}_was_missing"] = missing.astype("int8")
    result[f"{column}_is_not_applicable"] = is_na.astype("int8")
    result[f"{column}_status"] = status
    return result


def add_nullable_categorical_status(dataframe, column):
    """
    Giữ nguyên NA ở categorical cần theo dõi coverage (enhanced severity),
    không ép thành một nhóm quan sát giả.
    """
    result = dataframe.copy()
    missing = result[column].isna()
    status = pd.Series(OBSERVED_LABEL, index=result.index, dtype="string")
    status = status.mask(missing, UNKNOWN_LABEL)
    result[f"{column}_was_missing"] = missing.astype("int8")
    result[f"{column}_status"] = status
    return result


def prepare_collision_for_viz(dataframe):
    result = dataframe.drop(columns=COLLISION_DROP_COLUMNS, errors="ignore").copy()

    # Quy tắc rõ nhất: collision không ở junction (junction_detail == 0)
    # thì junction_control không áp dụng nếu nguồn để trống.
    no_junction = result["junction_detail"].eq(0)
    result = apply_categorical_missing_reason(
        result, "junction_control", not_applicable_mask=no_junction
    )

    result = fill_unknown_categorical(result, COLLISION_UNKNOWN_CATEGORICAL)

    # Numeric: giữ NA, không median/mean.
    result = add_numeric_missing_reason(result, "speed_limit")

    # Enhanced severity: giữ NA để coverage được minh bạch.
    result = add_nullable_categorical_status(result, "enhanced_severity_collision")
    return result


def prepare_vehicle_for_viz(dataframe):
    result = dataframe.drop(columns=VEHICLE_DROP_COLUMNS, errors="ignore").copy()

    # Dùng raw code trước khi vehicle_type được chuyển thành chuỗi/Unknown.
    is_pedal_cycle = result["vehicle_type"].eq(1)
    is_escooter = (
        result["escooter_flag"].eq(1)
        if "escooter_flag" in result.columns
        else pd.Series(False, index=result.index)
    )
    no_cc_or_registration_age = is_pedal_cycle | is_escooter

    # Propulsion của pedal cycle là N/A; e-scooter vẫn có hệ động lực nên không coi N/A.
    result = apply_categorical_missing_reason(
        result, "propulsion_code", not_applicable_mask=is_pedal_cycle
    )

    result = fill_unknown_categorical(result, VEHICLE_UNKNOWN_CATEGORICAL)

    result = add_numeric_missing_reason(result, "age_of_driver")
    result = add_numeric_missing_reason(
        result, "engine_capacity_cc", not_applicable_mask=no_cc_or_registration_age
    )
    result = add_numeric_missing_reason(
        result, "age_of_vehicle", not_applicable_mask=no_cc_or_registration_age
    )
    return result


def prepare_casualty_for_viz(dataframe):
    result = fill_unknown_categorical(dataframe, CASUALTY_UNKNOWN_CATEGORICAL)
    result = add_numeric_missing_reason(result, "age_of_casualty")
    result = add_nullable_categorical_status(result, "enhanced_casualty_severity")
    return result


## 5. Xử lý dữ liệu thiếu riêng cho từng bảng

Bảng hồ sơ bên dưới được tính **trước xử lý** để thấy mức độ thiếu thật sự. Sau đó notebook áp dụng các rule đã định nghĩa ở trên.

Lưu ý quan trọng:

- `second_road_class` / `second_road_number` **không** được tự động gán `Không áp dụng` chỉ vì `junction_detail = 0`, vì dữ liệu thực tế vẫn có nhiều giá trị quan sát ở nhóm này.
- `special_conditions_at_site` được loại khỏi bảng harmonized, không bị biến thành hàng chục nghìn giá trị `Không xác định`.
- `enhanced_*_severity` giữ `NA` và có cột status để biểu đồ có thể hiển thị coverage.


In [7]:
ATTRIBUTE_MEANINGS = {
    # Collision
    "local_authority_district": "Mã đơn vị hành chính theo hệ thống cũ",
    "junction_detail_historic": "Loại nút giao theo hệ mã cũ",
    "pedestrian_crossing_physical_facilities_historic": "Cơ sở vật chất qua đường theo hệ mã cũ",
    "carriageway_hazards_historic": "Nguy cơ trên lòng đường theo hệ mã cũ",
    "special_conditions_at_site": "Điều kiện đặc biệt tại hiện trường",
    "pedestrian_crossing_human_control_historic": "Kiểm soát người qua đường theo hệ mã cũ",
    "junction_control": "Hình thức điều khiển nút giao",
    "enhanced_severity_collision": "Mức độ nghiêm trọng tăng cường của vụ va chạm",
    "second_road_number": "Số hiệu tuyến đường thứ hai",
    "second_road_class": "Phân hạng tuyến đường thứ hai",
    "trunk_road_flag": "Vụ va chạm có nằm trên đường trục hay không",
    "junction_detail": "Loại nút giao hiện hành",
    "lsoa_of_accident_location": "Mã khu vực địa lý LSOA của vụ va chạm",
    "pedestrian_crossing": "Loại công trình hoặc vị trí qua đường",
    "road_surface_conditions": "Tình trạng bề mặt đường",
    "carriageway_hazards": "Nguy cơ trên lòng đường",
    "light_conditions": "Điều kiện ánh sáng",
    "speed_limit": "Giới hạn tốc độ, đơn vị mph",
    # Vehicle
    "vehicle_manoeuvre_historic": "Thao tác phương tiện theo hệ mã cũ",
    "journey_purpose_of_driver_historic": "Mục đích hành trình theo hệ mã cũ",
    "vehicle_location_restricted_lane_historic": "Vị trí trong làn hạn chế theo hệ mã cũ",
    "engine_capacity_cc": "Dung tích động cơ, đơn vị cc",
    "propulsion_code": "Loại nhiên liệu hoặc hệ động lực",
    "age_of_vehicle": "Tuổi phương tiện/tuổi đăng ký có sẵn trong nguồn",
    "generic_make_model": "Hãng và mẫu phương tiện",
    "lsoa_of_driver": "Mã khu vực địa lý LSOA của tài xế",
    "driver_imd_decile": "Nhóm mức độ thiếu thốn IMD của tài xế, từ 1 đến 10",
    "driver_distance_banding": "Nhóm khoảng cách của tài xế",
    "age_of_driver": "Tuổi tài xế",
    "age_band_of_driver": "Nhóm tuổi tài xế",
    "vehicle_manoeuvre": "Thao tác hoặc chuyển động của phương tiện",
    "vehicle_direction_to": "Hướng phương tiện đi tới",
    "vehicle_direction_from": "Hướng phương tiện xuất phát",
    "sex_of_driver": "Giới tính tài xế",
    "first_point_of_impact": "Điểm va chạm đầu tiên trên phương tiện",
    "vehicle_location_restricted_lane": "Vị trí phương tiện trong làn hạn chế",
    "vehicle_leaving_carriageway": "Cách phương tiện rời phần đường",
    "skidding_and_overturning": "Tình trạng trượt hoặc lật phương tiện",
    "hit_object_in_carriageway": "Vật thể bị đâm trên phần đường",
    "towing_and_articulation": "Tình trạng kéo moóc hoặc khớp nối",
    "journey_purpose_of_driver": "Mục đích hành trình của tài xế",
    "vehicle_type": "Loại phương tiện",
    "junction_location": "Vị trí phương tiện tại nút giao",
    "hit_object_off_carriageway": "Vật thể bị đâm ngoài phần đường",
    "vehicle_left_hand_drive": "Phương tiện có tay lái bên trái hay không",
    # Casualty
    "enhanced_casualty_severity": "Mức độ thương tích tăng cường",
    "lsoa_of_casualty": "Mã khu vực địa lý LSOA của người thương vong",
    "casualty_imd_decile": "Nhóm mức độ thiếu thốn IMD của người thương vong",
    "casualty_distance_banding": "Nhóm khoảng cách của người thương vong",
    "pedestrian_road_maintenance_worker": "Trạng thái người đi bộ/công nhân bảo trì đường",
    "age_of_casualty": "Tuổi người thương vong",
    "age_band_of_casualty": "Nhóm tuổi người thương vong",
    "sex_of_casualty": "Giới tính người thương vong",
    "casualty_type": "Loại người thương vong hoặc phương tiện liên quan",
    "car_passenger": "Trạng thái hoặc vị trí hành khách ô tô",
    "bus_or_coach_passenger": "Trạng thái hành khách xe buýt hoặc xe khách",
    "pedestrian_movement": "Hướng/chuyển động của người đi bộ",
    "pedestrian_location": "Vị trí của người đi bộ",
}

SPECIAL_TREATMENT = {
    # Collision
    "local_authority_district": "Drop khỏi dataset viz: thiếu hoàn toàn/biến cũ",
    "junction_detail_historic": "Drop: cột historic",
    "pedestrian_crossing_human_control_historic": "Drop: cột historic",
    "pedestrian_crossing_physical_facilities_historic": "Drop: cột historic",
    "carriageway_hazards_historic": "Drop: cột historic",
    "special_conditions_at_site": "Drop khỏi bảng harmonized; không nội suy",
    "junction_control": "NA + junction_detail=0 → Không áp dụng; NA còn lại → Không xác định",
    "speed_limit": "Giữ NA + status; không nội suy",
    "enhanced_severity_collision": "Giữ NA + status để theo dõi coverage",
    # Vehicle
    "vehicle_manoeuvre_historic": "Drop: cột historic",
    "vehicle_location_restricted_lane_historic": "Drop: cột historic",
    "journey_purpose_of_driver_historic": "Drop: cột historic",
    "age_of_driver": "Giữ NA + status; không nội suy",
    "engine_capacity_cc": "Giữ NA; pedal cycle/e-scooter thiếu → Không áp dụng; còn lại → Không xác định",
    "propulsion_code": "Pedal cycle thiếu → Không áp dụng; còn lại → Không xác định",
    "age_of_vehicle": "Giữ NA; pedal cycle/e-scooter thiếu → Không áp dụng; còn lại → Không xác định",
    # Casualty
    "age_of_casualty": "Giữ NA + status; không nội suy",
    "enhanced_casualty_severity": "Giữ NA + status để theo dõi coverage",
}


def default_treatment(table_name, column):
    if table_name == "Collision" and column in COLLISION_UNKNOWN_CATEGORICAL:
        return f"NA → '{UNKNOWN_LABEL}'"
    if table_name == "Vehicle" and column in VEHICLE_UNKNOWN_CATEGORICAL:
        return f"NA → '{UNKNOWN_LABEL}'"
    if table_name == "Casualty" and column in CASUALTY_UNKNOWN_CATEGORICAL:
        return f"NA → '{UNKNOWN_LABEL}'"
    return "Giữ NA và kiểm tra theo ngữ cảnh"


def build_missing_profile(table_name, dataframe):
    rows = []
    for column in dataframe.columns:
        missing_count = int(dataframe[column].isna().sum())
        if missing_count == 0:
            continue
        rows.append({
            "Bảng": table_name,
            "Thuộc tính": column,
            "Số thiếu": missing_count,
            "Tỷ lệ thiếu (%)": round(100 * missing_count / len(dataframe), 2),
            "Số giá trị quan sát": int(dataframe[column].nunique(dropna=True)),
            "Ý nghĩa": ATTRIBUTE_MEANINGS.get(column, "Chưa bổ sung mô tả"),
            "Xử lý": SPECIAL_TREATMENT.get(
                column, default_treatment(table_name, column)
            ),
        })
    return rows


missing_profile = pd.DataFrame(
    build_missing_profile("Collision", collision)
    + build_missing_profile("Vehicle", vehicle)
    + build_missing_profile("Casualty", casualty)
).sort_values(
    ["Bảng", "Tỷ lệ thiếu (%)"], ascending=[True, False]
).reset_index(drop=True)

print("Hồ sơ dữ liệu thiếu TRƯỚC xử lý:")
display(missing_profile)

collision_before = collision.copy()
vehicle_before = vehicle.copy()
casualty_before = casualty.copy()

collision = prepare_collision_for_viz(collision)
vehicle = prepare_vehicle_for_viz(vehicle)
casualty = prepare_casualty_for_viz(casualty)

# Tóm tắt các reason quan trọng sau xử lý
reason_summary = pd.DataFrame({
    "Biến": [
        "junction_control",
        "engine_capacity_cc",
        "propulsion_code",
        "age_of_vehicle",
        "enhanced_severity_collision",
        "enhanced_casualty_severity",
    ],
    "Không áp dụng": [
        int(collision["junction_control_status"].eq(NOT_APPLICABLE_LABEL).sum()),
        int(vehicle["engine_capacity_cc_status"].eq(NOT_APPLICABLE_LABEL).sum()),
        int(vehicle["propulsion_code_status"].eq(NOT_APPLICABLE_LABEL).sum()),
        int(vehicle["age_of_vehicle_status"].eq(NOT_APPLICABLE_LABEL).sum()),
        0,
        0,
    ],
    "Không xác định": [
        int(collision["junction_control_status"].eq(UNKNOWN_LABEL).sum()),
        int(vehicle["engine_capacity_cc_status"].eq(UNKNOWN_LABEL).sum()),
        int(vehicle["propulsion_code_status"].eq(UNKNOWN_LABEL).sum()),
        int(vehicle["age_of_vehicle_status"].eq(UNKNOWN_LABEL).sum()),
        int(collision["enhanced_severity_collision_status"].eq(UNKNOWN_LABEL).sum()),
        int(casualty["enhanced_casualty_severity_status"].eq(UNKNOWN_LABEL).sum()),
    ],
})
display(reason_summary)

print("Số dòng được giữ nguyên sau xử lý:")
display(pd.DataFrame({
    "Bảng": ["Collision", "Vehicle", "Casualty"],
    "Trước": [len(collision_before), len(vehicle_before), len(casualty_before)],
    "Sau": [len(collision), len(vehicle), len(casualty)],
}))


Hồ sơ dữ liệu thiếu TRƯỚC xử lý:


,Bảng,Thuộc tính,Số thiếu,Tỷ lệ thiếu (%),Số giá trị quan sát,Ý nghĩa,Xử lý
0,Casualty,enhanced_casualty_severity,26789,20.88,5,Mức độ thương tích tăng cường,Giữ NA + status để theo dõi coverage
1,Casualty,lsoa_of_casualty,21646,16.88,30899,Mã khu vực địa lý LSOA của người thương vong,NA → 'Không xác định'
2,Casualty,casualty_imd_decile,17275,13.47,10,Nhóm mức độ thiếu thốn IMD của người thương vong,NA → 'Không xác định'
3,Casualty,casualty_distance_banding,17275,13.47,5,Nhóm khoảng cách của người thương vong,NA → 'Không xác định'
4,Casualty,pedestrian_road_maintenance_worker,11233,8.76,3,Trạng thái người đi bộ/công nhân bảo trì đường,NA → 'Không xác định'
5,Casualty,age_of_casualty,2552,1.99,100,Tuổi người thương vong,Giữ NA + status; không nội suy
6,Casualty,age_band_of_casualty,2552,1.99,11,Nhóm tuổi người thương vong,NA → 'Không xác định'
7,Casualty,sex_of_casualty,1412,1.10,3,Giới tính người thương vong,NA → 'Không xác định'
8,Casualty,casualty_type,1120,0.87,22,Loại người thương vong hoặc phương tiện liên quan,NA → 'Không xác định'
9,Casualty,car_passenger,190,0.15,4,Trạng thái hoặc vị trí hành khách ô tô,NA → 'Không xác định'


,Biến,Không áp dụng,Không xác định
0,junction_control,41637,944
1,engine_capacity_cc,16683,37714
2,propulsion_code,15362,35102
3,age_of_vehicle,16683,33759
4,enhanced_severity_collision,0,41630
5,enhanced_casualty_severity,0,26789


Số dòng được giữ nguyên sau xử lý:


,Bảng,Trước,Sau
0,Collision,100927,100927
1,Vehicle,183514,183514
2,Casualty,128272,128272


## 6. Kiểm tra khóa chính và quan hệ

- `collision_index`: khóa duy nhất của bảng collision.
- `collision_index + vehicle_reference`: khóa ghép duy nhất của bảng vehicle.
- `collision_index + vehicle_reference + casualty_reference`: khóa ghép duy nhất của bảng casualty.

In [8]:
key_checks = {
    "collision_index_is_unique": not collision.duplicated(["collision_index"]).any(),
    "vehicle_key_is_unique": not vehicle.duplicated(
        ["collision_index", "vehicle_reference"]
    ).any(),
    "casualty_key_is_unique": not casualty.duplicated(
        ["collision_index", "vehicle_reference", "casualty_reference"]
    ).any(),
}

casualty_vehicle_match = casualty.merge(
    vehicle[["collision_index", "vehicle_reference"]],
    how="left",
    on=["collision_index", "vehicle_reference"],
    indicator=True,
    validate="many_to_one",
)
relation_checks = pd.DataFrame({
    "Kiểm tra": [
        "collision_index là duy nhất",
        "Khóa vehicle là duy nhất",
        "Khóa casualty là duy nhất",
        "Vehicle không tìm thấy collision",
        "Casualty không tìm thấy vehicle",
    ],
    "Kết quả": [
        key_checks["collision_index_is_unique"],
        key_checks["vehicle_key_is_unique"],
        key_checks["casualty_key_is_unique"],
        int((~vehicle["collision_index"].isin(collision["collision_index"])).sum()),
        int(casualty_vehicle_match["_merge"].eq("left_only").sum()),
    ],
})
display(relation_checks)

,Kiểm tra,Kết quả
0,collision_index là duy nhất,True
1,Khóa vehicle là duy nhất,True
2,Khóa casualty là duy nhất,True
3,Vehicle không tìm thấy collision,0
4,Casualty không tìm thấy vehicle,0


## 7. Gộp ba bảng và tạo cờ đếm an toàn cho BI

Gộp trái từ `vehicle` sang `casualty` giúp giữ cả phương tiện không có người thương vong. Các `NA` ở nhóm cột casualty sinh ra trên những dòng này là **merge-generated missing**, không phải dữ liệu casualty bị thất lạc.

`record_key` là khóa duy nhất của từng dòng đầu ra:

- Dòng có casualty: dùng `casualty_key`.
- Vehicle không có casualty: dùng `vehicle_key`.

Để tránh double counting trong Power BI/Tableau, notebook tạo thêm:

- `collision_count_flag`: tổng bằng số collision duy nhất.
- `vehicle_count_flag`: tổng bằng số vehicle duy nhất.
- `casualty_count_flag`: tổng bằng số casualty.
- `row_count_flag`: luôn bằng 1, chỉ dùng khi thực sự cần đếm dòng ở grain cuối.

**Không dùng `SUM(number_of_vehicles)` hoặc `SUM(number_of_casualties)` trên bảng đã gộp.**


In [9]:
vehicle_payload = vehicle.drop(
    columns=["collision_year", "collision_ref_no"], errors="ignore"
)
casualty_payload = casualty.drop(
    columns=["collision_year", "collision_ref_no"], errors="ignore"
)

vehicle_casualty = vehicle_payload.merge(
    casualty_payload,
    how="left",
    on=["collision_index", "vehicle_reference"],
    validate="one_to_many",
    indicator="_casualty_merge",
)
vehicle_casualty["has_casualty_record"] = vehicle_casualty["_casualty_merge"].eq("both")
vehicle_casualty["record_type"] = np.where(
    vehicle_casualty["has_casualty_record"],
    "casualty_record",
    "vehicle_without_casualty",
)
vehicle_casualty["casualty_data_status"] = np.where(
    vehicle_casualty["has_casualty_record"],
    OBSERVED_LABEL,
    "Không áp dụng - vehicle không có casualty",
)

vehicle_casualty["vehicle_key"] = (
    vehicle_casualty["collision_index"].astype("string")
    + "_V"
    + vehicle_casualty["vehicle_reference"].astype("Int64").astype("string")
)
vehicle_casualty["casualty_key"] = (
    vehicle_casualty["vehicle_key"]
    + "_C"
    + vehicle_casualty["casualty_reference"].astype("Int64").astype("string")
).where(vehicle_casualty["has_casualty_record"], pd.NA)
vehicle_casualty["record_key"] = vehicle_casualty["casualty_key"].fillna(
    vehicle_casualty["vehicle_key"]
)
vehicle_casualty = vehicle_casualty.drop(columns="_casualty_merge")

merged = collision.merge(
    vehicle_casualty,
    how="left",
    on="collision_index",
    validate="one_to_many",
    indicator="_collision_merge",
)
if not merged["_collision_merge"].eq("both").all():
    raise ValueError("Có collision không tìm thấy vehicle khi gộp dữ liệu.")
merged = merged.drop(columns="_collision_merge")

# Grain-aware flags: an toàn hơn COUNTROWS/SUM các biến cấp collision.
merged["collision_count_flag"] = (
    ~merged.duplicated(["collision_index"])
).astype("int8")
merged["vehicle_count_flag"] = (
    ~merged.duplicated(["collision_index", "vehicle_reference"])
).astype("int8")
merged["casualty_count_flag"] = merged["has_casualty_record"].astype("int8")
merged["row_count_flag"] = np.int8(1)

front_columns = [
    "record_key", "collision_index", "collision_year", "collision_ref_no",
    "vehicle_key", "vehicle_reference", "casualty_key", "casualty_reference",
    "record_type", "has_casualty_record", "casualty_data_status",
    "collision_count_flag", "vehicle_count_flag", "casualty_count_flag", "row_count_flag",
    "date", "time", "collision_month", "collision_quarter",
    "collision_day_of_month", "collision_hour", "is_weekend",
]
front_columns = [column for column in front_columns if column in merged.columns]
remaining_columns = [column for column in merged.columns if column not in front_columns]
merged = merged[front_columns + remaining_columns]

print(f"Kết quả gộp: {merged.shape[0]:,} dòng × {merged.shape[1]:,} cột")
display(merged.head())


Kết quả gộp: 200,522 dòng × 121 cột


,record_key,collision_index,collision_year,collision_ref_no,vehicle_key,vehicle_reference,casualty_key,casualty_reference,record_type,has_casualty_record,casualty_data_status,collision_count_flag,vehicle_count_flag,casualty_count_flag,row_count_flag,date,time,collision_month,collision_quarter,collision_day_of_month,collision_hour,is_weekend,location_easting_osgr,location_northing_osgr,longitude,latitude,police_force,collision_severity,number_of_vehicles,number_of_casualties,day_of_week,local_authority_ons_district,local_authority_highway,local_authority_highway_current,first_road_class,first_road_number,road_type,speed_limit,junction_detail,junction_control,second_road_class,second_road_number,pedestrian_crossing,light_conditions,weather_conditions,road_surface_conditions,carriageway_hazards,urban_or_rural_area,did_police_officer_attend_scene_of_accident,trunk_road_flag,lsoa_of_accident_location,enhanced_severity_collision,collision_injury_based,collision_adjusted_severity_serious,collision_adjusted_severity_slight,junction_control_status,junction_control_was_missing,speed_limit_was_missing,speed_limit_is_not_applicable,speed_limit_status,enhanced_severity_collision_was_missing,enhanced_severity_collision_status,vehicle_type,towing_and_articulation,vehicle_manoeuvre,vehicle_direction_from,vehicle_direction_to,vehicle_location_restricted_lane,junction_location,skidding_and_overturning,hit_object_in_carriageway,vehicle_leaving_carriageway,hit_object_off_carriageway,first_point_of_impact,vehicle_left_hand_drive,journey_purpose_of_driver,sex_of_driver,age_of_driver,age_band_of_driver,engine_capacity_cc,propulsion_code,age_of_vehicle,generic_make_model,driver_imd_decile,lsoa_of_driver,escooter_flag,driver_distance_banding,propulsion_code_status,propulsion_code_was_missing,age_of_driver_was_missing,age_of_driver_is_not_applicable,age_of_driver_status,engine_capacity_cc_was_missing,engine_capacity_cc_is_not_applicable,engine_capacity_cc_status,age_of_vehicle_was_missing,age_of_vehicle_is_not_applicable,age_of_vehicle_status,casualty_class,sex_of_casualty,age_of_casualty,age_band_of_casualty,casualty_severity,pedestrian_location,pedestrian_movement,car_passenger,bus_or_coach_passenger,pedestrian_road_maintenance_worker,casualty_type,casualty_imd_decile,lsoa_of_casualty,enhanced_casualty_severity,casualty_injury_based,casualty_adjusted_severity_serious,casualty_adjusted_severity_slight,casualty_distance_banding,age_of_casualty_was_missing,age_of_casualty_is_not_applicable,age_of_casualty_status,enhanced_casualty_severity_was_missing,enhanced_casualty_severity_status
0,202417S111924_V1_C1,202417S111924,2024,17S111924,202417S111924_V1,1,202417S111924_V1_C1,1,casualty_record,True,Có dữ liệu,1,1,1,1,2024-10-22,14:48,10,Q4,22,14,False,444670,520502,-1.31046,54.57776,17,2,1,1,3,E06000004,E06000004,E06000004,3,139,6,30,0,Không áp dụng,Không xác định,Không xác định,0,1,1,1,0,1,1,2,E01012194,7,1,1.0,0.0,Không áp dụng,1,0,0,Có dữ liệu,0,Có dữ liệu,9,0,19,5,1,0,0,0,0,0,0,4,1,9,2,62,9,1461,2,14,NISSAN NOTE,9,E01012199,0,2,Có dữ liệu,0,0,0,Có dữ liệu,0,0,Có dữ liệu,0,0,Có dữ liệu,3,1,12,3,2,5,1,0,0,Không xác định,0,2,E01012193,7,1,1.0,0.0,1,0.0,0.0,Có dữ liệu,0.0,Có dữ liệu
1,2024070259500_V2_C2,2024070259500,2024,070259500,2024070259500_V2,2,2024070259500_V2_C2,2,casualty_record,True,Có dữ liệu,1,1,1,1,2024-03-16,10:45,3,Q1,16,10,True,364248,391007,-2.53935,53.41465,7,3,2,3,7,E06000007,E06000007,E06000007,1,6,3,70,0,4,1,62,0,1,1,1,0,2,2,1,E01012463,<NA>,0,0.08418,0.91582,Có dữ liệu,0,0,0,Có dữ liệu,1,Không xác định,9,0,19,8,4,0,1,0,0,5,6,1,1,6,2,18,4,1368,1,14,ALFA ROMEO MITO,4,E01025543,0,4,Có dữ liệu,0,0,0,Có dữ liệu,0,0,Có dữ liệu,0,0,Có dữ liệu,2,2,38,7,3,0,0,1,0,0,9,4,E01025543,<NA>,0,0.03308,0.96692,4,0.0,0.0,Có dữ liệu,1.0,Không xác định
2,2024070259500_V2_C3,2024070259500,2024,070259500,2024070259500_V2,2,2024070259500_V2_C3,3,casualty_record,True,Có dữ liệu,0,0,1,1,2024-03-16,10:45,3,Q1,16,10,True,364248,391007,-2.53935,53.41465,7,3,2,3,7

## 8. Đối soát dữ liệu sau xử lý và sau gộp

Các kiểm tra bên dưới xác nhận:

- Không mất collision, vehicle hoặc casualty.
- `record_key` duy nhất.
- Các cờ đếm BI cho đúng tổng ở từng grain.
- `Unknown` và `Not applicable` được tách đúng ở các rule chính.
- Numeric missing vẫn là `NA` và khớp với `_was_missing`.
- Enhanced severity vẫn giữ `NA`, không bị “lấp” để tạo coverage giả.


In [10]:
vehicles_without_casualties = int((~vehicle_casualty["has_casualty_record"]).sum())
expected_rows = len(casualty) + vehicles_without_casualties
vehicle_pairs_in_output = merged[["collision_index", "vehicle_reference"]].drop_duplicates()
casualty_rows_in_output = int(merged["has_casualty_record"].sum())

# Các categorical đã quyết định fill bằng label phải hết NA.
filled_categorical_checks = {
    "collision_unknown_categories": int(
        collision[COLLISION_UNKNOWN_CATEGORICAL].isna().sum().sum()
    ) == 0,
    "junction_control": int(collision["junction_control"].isna().sum()) == 0,
    "vehicle_unknown_categories": int(
        vehicle[VEHICLE_UNKNOWN_CATEGORICAL].isna().sum().sum()
    ) == 0,
    "propulsion_code": int(vehicle["propulsion_code"].isna().sum()) == 0,
    "casualty_unknown_categories": int(
        casualty[CASUALTY_UNKNOWN_CATEGORICAL].isna().sum().sum()
    ) == 0,
}

numeric_flag_checks = {
    "collision_speed_limit": int(collision["speed_limit"].isna().sum())
    == int(collision["speed_limit_was_missing"].sum()),
    "vehicle_age_of_driver": int(vehicle["age_of_driver"].isna().sum())
    == int(vehicle["age_of_driver_was_missing"].sum()),
    "vehicle_engine_capacity_cc": int(vehicle["engine_capacity_cc"].isna().sum())
    == int(vehicle["engine_capacity_cc_was_missing"].sum()),
    "vehicle_age_of_vehicle": int(vehicle["age_of_vehicle"].isna().sum())
    == int(vehicle["age_of_vehicle_was_missing"].sum()),
    "casualty_age_of_casualty": int(casualty["age_of_casualty"].isna().sum())
    == int(casualty["age_of_casualty_was_missing"].sum()),
}

# Kiểm tra quy tắc N/A chính.
junction_na_expected = collision_before["junction_control"].isna() & collision_before["junction_detail"].eq(0)
junction_na_actual = collision["junction_control_status"].eq(NOT_APPLICABLE_LABEL)

is_pedal_cycle_before = vehicle_before["vehicle_type"].eq(1)
is_escooter_before = (
    vehicle_before["escooter_flag"].eq(1)
    if "escooter_flag" in vehicle_before.columns
    else pd.Series(False, index=vehicle_before.index)
)
engine_na_expected = (
    vehicle_before["engine_capacity_cc"].isna()
    & (is_pedal_cycle_before | is_escooter_before)
)
age_vehicle_na_expected = (
    vehicle_before["age_of_vehicle"].isna()
    & (is_pedal_cycle_before | is_escooter_before)
)
propulsion_na_expected = (
    vehicle_before["propulsion_code"].isna()
    & is_pedal_cycle_before
)

def same_boolean_values(left, right):
    """So sánh giá trị boolean, bỏ qua metadata như Series.name."""
    return np.array_equal(
        left.fillna(False).to_numpy(dtype=bool),
        right.fillna(False).to_numpy(dtype=bool),
    )


reason_rule_checks = {
    "junction_control_NA_rule": same_boolean_values(
        junction_na_actual, junction_na_expected
    ),
    "engine_capacity_NA_rule": same_boolean_values(
        vehicle["engine_capacity_cc_status"].eq(NOT_APPLICABLE_LABEL),
        engine_na_expected,
    ),
    "age_of_vehicle_NA_rule": same_boolean_values(
        vehicle["age_of_vehicle_status"].eq(NOT_APPLICABLE_LABEL),
        age_vehicle_na_expected,
    ),
    "propulsion_NA_rule": same_boolean_values(
        vehicle["propulsion_code_status"].eq(NOT_APPLICABLE_LABEL),
        propulsion_na_expected,
    ),
}

# Merge-generated missing: vehicle không casualty phải được nhận diện bằng status,
# không được biến thành một casualty "Không xác định".
vehicle_without_cas_mask = ~merged["has_casualty_record"]
merge_missing_checks = {
    "casualty_status_on_vehicle_without_casualty": merged.loc[
        vehicle_without_cas_mask, "casualty_data_status"
    ].eq("Không áp dụng - vehicle không có casualty").all(),
    "casualty_reference_blank_when_no_casualty": merged.loc[
        vehicle_without_cas_mask, "casualty_reference"
    ].isna().all(),
}

validation_summary = pd.DataFrame({
    "Chỉ tiêu": [
        "Collision đầu vào",
        "Vehicle đầu vào",
        "Casualty đầu vào",
        "Vehicle không có casualty",
        "Số dòng đầu ra dự kiến",
        "Số dòng đầu ra thực tế",
        "Số collision trong đầu ra",
        "SUM(collision_count_flag)",
        "Số vehicle trong đầu ra",
        "SUM(vehicle_count_flag)",
        "Số casualty trong đầu ra",
        "SUM(casualty_count_flag)",
        "Khóa record_key bị thiếu",
        "Khóa record_key bị trùng",
        "Enhanced collision severity còn NA",
        "Enhanced casualty severity còn NA",
    ],
    "Giá trị": [
        len(collision), len(vehicle), len(casualty),
        vehicles_without_casualties, expected_rows, len(merged),
        merged["collision_index"].nunique(),
        int(merged["collision_count_flag"].sum()),
        len(vehicle_pairs_in_output),
        int(merged["vehicle_count_flag"].sum()),
        casualty_rows_in_output,
        int(merged["casualty_count_flag"].sum()),
        int(merged["record_key"].isna().sum()),
        int(merged["record_key"].duplicated().sum()),
        int(collision["enhanced_severity_collision"].isna().sum()),
        int(casualty["enhanced_casualty_severity"].isna().sum()),
    ],
})
display(validation_summary)

print("Kiểm tra categorical:")
display(pd.Series(filled_categorical_checks, name="Đạt"))
print("Kiểm tra numeric missing flags:")
display(pd.Series(numeric_flag_checks, name="Đạt"))
print("Kiểm tra rule Unknown / Not applicable:")
display(pd.Series(reason_rule_checks, name="Đạt"))
print("Kiểm tra merge-generated missing:")
display(pd.Series(merge_missing_checks, name="Đạt"))

assert all(key_checks.values())
assert len(merged) == expected_rows
assert merged["collision_index"].nunique() == len(collision)
assert len(vehicle_pairs_in_output) == len(vehicle)
assert casualty_rows_in_output == len(casualty)
assert merged["record_key"].notna().all()
assert not merged["record_key"].duplicated().any()

assert int(merged["collision_count_flag"].sum()) == len(collision)
assert int(merged["vehicle_count_flag"].sum()) == len(vehicle)
assert int(merged["casualty_count_flag"].sum()) == len(casualty)

assert all(filled_categorical_checks.values())
assert all(numeric_flag_checks.values())
assert all(reason_rule_checks.values())
assert all(merge_missing_checks.values())

print("Tất cả kiểm tra đối soát đều đạt.")


,Chỉ tiêu,Giá trị
0,Collision đầu vào,100927
1,Vehicle đầu vào,183514
2,Casualty đầu vào,128272
3,Vehicle không có casualty,72250
4,Số dòng đầu ra dự kiến,200522
5,Số dòng đầu ra thực tế,200522
6,Số collision trong đầu ra,100927
7,SUM(collision_count_flag),100927
8,Số vehicle trong đầu ra,183514
9,SUM(vehicle_count_flag),183514


Kiểm tra categorical:


collision_unknown_categories    True
junction_control                True
vehicle_unknown_categories      True
propulsion_code                 True
casualty_unknown_categories     True
Name: Đạt, dtype: bool

Kiểm tra numeric missing flags:


collision_speed_limit         True
vehicle_age_of_driver         True
vehicle_engine_capacity_cc    True
vehicle_age_of_vehicle        True
casualty_age_of_casualty      True
Name: Đạt, dtype: bool

Kiểm tra rule Unknown / Not applicable:


junction_control_NA_rule    True
engine_capacity_NA_rule     True
age_of_vehicle_NA_rule      True
propulsion_NA_rule          True
Name: Đạt, dtype: bool

Kiểm tra merge-generated missing:


casualty_status_on_vehicle_without_casualty    True
casualty_reference_blank_when_no_casualty      True
Name: Đạt, dtype: bool

Tất cả kiểm tra đối soát đều đạt.


## 9. Xuất một file dữ liệu duy nhất

File CSV dùng UTF-8 có BOM để mở thuận tiện trong Excel/Power BI.

Quy ước đầu ra:

- `Không xác định`: nguồn thiếu/no data ở biến categorical.
- `Không áp dụng`: thuộc tính categorical không áp dụng theo rule rõ ràng.
- Numeric không có giá trị vẫn để ô trống; đọc cột `*_status` để biết **Unknown** hay **Not applicable**.
- Casualty fields ở `vehicle_without_casualty` vẫn để trống và được giải thích bằng `casualty_data_status`.
- Đếm collision/vehicle/casualty bằng các cột `*_count_flag`; không cộng trực tiếp biến cấp collision sau khi join.


In [11]:
merged.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig",
    date_format="%Y-%m-%d",
    na_rep="",
)

file_size_mb = OUTPUT_CSV.stat().st_size / (1024 ** 2)
print(f"Đã xuất: {OUTPUT_CSV.name}")
print(f"Dung lượng: {file_size_mb:,.1f} MB")

Đã xuất: du_lieu_tai_nan_2024_san_sang_truc_quan.csv
Dung lượng: 121.3 MB
